In [1]:
import os
import sys

# add the source directory to system path, so that relative imports work
# this fix is only for Jupyter Notebooks
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pykep

from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

from orbital_mechanics.solar_system import SolarSystem
from common.constants import ALTAIRA_MU as MU, ALTAIRA_AU as AU, YEAR

In [4]:
ss = SolarSystem()

# get PlanetX
planetX = [pl for pl in ss.bodies if pl.name == 'PlanetX'][0]

# spacecraft initial state at t0
rv0 = np.zeros((6,))
rv0[0] = -200*AU
print(planetX)

Id: 10
Type: planet
Weight: 50.0
Planet Name: PlanetX
Own gravity parameter: 3411912.3969999999
Central body gravity parameter: 139348062043.34299
Planet radius: 12993.799999999999
Planet safe radius: 14293.18
Keplerian planet elements: 
Semi major axis (AU): 0.10665270041384937
Eccentricity: 0.34499999999999997
Inclination (deg.): 40
Big Omega (deg.): 341.69299999999998
Small omega (deg.): 29.999999999999996
Mean anomaly (deg.): 133.42699999999999
Elements reference epoch: 2000-Jan-01 00:00:00
Ephemerides type: Keplerian
r at ref. = [-19776387679.20636, 5091000491.9670391, -1156782917.5597]
v at ref. = [-0.78096103990052712, -1.5132627939144223, -1.4113476643111722]



In [5]:
# dynamics function
def dydt(t,rv):
    r = rv[0:3]; v = rv[3:6]
    r_mag = np.linalg.norm(r)
    a = -MU/r_mag**3 * r
    dr = v; dv = a
    drv = np.concatenate([dr, dv])
    return drv

In [6]:
# distance from planetX
def rel_rv_from_X(t,rv):
    r = rv[0:3]; v = rv[3:6]

    # get planetX's position and velocity
    pl_r, pl_v = planetX.eph(t*pykep.SEC2DAY)

    rel_r = r - pl_r
    rel_v = v - pl_v

    return rel_r, rel_v

In [7]:
# event function (check periapsis distance to planetX)
def event(t,rv):
    rel_r, rel_v = rel_rv_from_X(t,rv)
    res = np.dot(rel_r, rel_v)

    return res

event.terminal = True

In [8]:
res = solve_ivp(dydt, [0, 200*YEAR], rv0, events=event, method='DOP853', rtol=1e-12, atol=1e-12)

In [9]:
vals = []
for i in range(len(res['t'])):
    rel_r, _ = rel_rv_from_X(res['t'][i],res['y'][:,i].ravel())
    rel_r_mag = np.linalg.norm(rel_r)
    dotp = event(res['t'][i], res['y'][:,i].ravel())

    vals.append([res['t'][i] / YEAR, rel_r_mag / AU, dotp])

vals = np.array(vals, dtype=float)
print(vals)

[[ 0.00000000e+00  7.62572426e+01 -1.39928322e+10]
 [ 1.06049035e-08  7.62572426e+01 -1.39928322e+10]
 [ 1.16653939e-07  7.62572426e+01 -1.39928322e+10]
 [ 1.17714429e-06  7.62572423e+01 -1.39928320e+10]
 [ 1.17820478e-05  7.62572396e+01 -1.39928299e+10]
 [ 1.17831083e-04  7.62572121e+01 -1.39928094e+10]
 [ 1.17832143e-03  7.62569377e+01 -1.39926041e+10]
 [ 1.17832249e-02  7.62541939e+01 -1.39905507e+10]
 [ 1.17832260e-01  7.62267725e+01 -1.39700175e+10]
 [ 1.17832261e+00  7.59542368e+01 -1.37647210e+10]
 [ 1.17832261e+01  7.34031276e+01 -1.17168691e+10]
 [ 5.62795911e+01  6.66868504e+01 -3.36410917e+09]
 [ 7.52786065e+01  6.60129310e+01 -1.90734863e-06]]


In [10]:
# root func
def rootfunc(y0):
    yy0 = np.array([-200*AU, y0[0], y0[1], y0[2], 0.0, 0.0])
    res = solve_ivp(dydt, [0, 70*YEAR], yy0, events=event, method='DOP853', rtol=1e-12, atol=1e-12)
    rel_r, _ = rel_rv_from_X(res['t'][-1],res['y'][:,-1].ravel())
    print(res['t'][-1] / YEAR, rel_r)
    return rel_r

In [14]:
res = least_squares(rootfunc, [-15.5*AU, -48*AU, 0.0], bounds=([-100*AU,-100*AU,0.0], [100*AU,100*AU,4]))

70.0 [-8.84022166e+09 -3.88877844e+09 -2.90522059e+09]
70.0 [-8.84022166e+09 -3.88877848e+09 -2.90522059e+09]
70.0 [-8.84022166e+09 -3.88877844e+09 -2.90522070e+09]
70.0 [-8.84022163e+09 -3.88877844e+09 -2.90522059e+09]
70.0 [-4.39359224e+09 -7.22902512e+08 -3.20466844e+08]
70.0 [-4.39359224e+09 -7.22902499e+08 -3.20466844e+08]
70.0 [-4.39359224e+09 -7.22902512e+08 -3.20466911e+08]
70.0 [-4.39359217e+09 -7.22902512e+08 -3.20466843e+08]
70.0 [-2.17046002e+09 -3.93870867e+07  9.13097012e+06]
70.0 [-2.17046002e+09 -3.93870635e+07  9.13097011e+06]
70.0 [-2.17046002e+09 -3.93870867e+07  9.13090772e+06]
70.0 [-2.17045992e+09 -3.93870868e+07  9.13097040e+06]
70.0 [-1.05911240e+09 -3.05274475e+06  7.86912353e+06]
70.0 [-1.05911240e+09 -3.05272100e+06  7.86912352e+06]
70.0 [-1.05911240e+09 -3.05274473e+06  7.86906111e+06]
70.0 [-1.05911228e+09 -3.05274489e+06  7.86912389e+06]
70.0 [-5.04061181e+08 -1.51209845e+06  3.96954713e+06]
70.0 [-5.04061181e+08 -1.51207468e+06  3.96954712e+06]
70.0 [-5.0

In [15]:
res['message']

'`xtol` termination condition is satisfied.'

In [16]:
human_res = res['x'].copy()
human_res[0] /= AU
human_res[1] /= AU

print(human_res)


[ 10.86263103 -28.51480415   3.95211844]


In [17]:
res = solve_ivp(dydt, [0, 70*YEAR],
                np.array([-200*AU, human_res[0]*AU, human_res[1]*AU, human_res[2], 0.0, 0.0]),
                events=event, method='DOP853', rtol=1e-12, atol=1e-12)

In [18]:
res['t'] / YEAR

array([0.00000000e+00, 1.07710068e-08, 1.18481075e-07, 1.19558176e-06,
       1.19665886e-05, 1.19676657e-04, 1.19677734e-03, 1.19677842e-02,
       1.19677852e-01, 1.19677853e+00, 1.19677854e+01, 2.92870044e+01,
       4.33577632e+01, 5.74285221e+01, 7.00000000e+01])

In [19]:
(res['y'][0:3]).T / AU

array([[-200.        ,   10.86263103,  -28.51480415],
       [-199.99999999,   10.86263103,  -28.51480415],
       [-199.9999999 ,   10.86263103,  -28.51480415],
       [-199.999999  ,   10.86263103,  -28.51480415],
       [-199.99999002,   10.86263103,  -28.51480415],
       [-199.99990023,   10.86263103,  -28.51480415],
       [-199.99900225,   10.86263103,  -28.51480415],
       [-199.99002242,   10.86263103,  -28.51480414],
       [-199.9002177 ,   10.86263064,  -28.51480312],
       [-199.00152946,   10.8625919 ,  -28.51470143],
       [-189.94839547,   10.85853708,  -28.50405738],
       [-175.11675133,   10.83613161,  -28.4452422 ],
       [-162.78357372,   10.80043233,  -28.35153028],
       [-150.15529232,   10.74510043,  -28.20628203],
       [-138.58320844,   10.67495418,  -28.02214553]])

In [20]:
rel_r, rel_v = rel_rv_from_X(res['t'][-1], res['y'][:,-1].T)

In [21]:
rel_r, rel_v

(array([-2.09426880e-03, -6.91413879e-06,  1.57356262e-05]),
 array([4.5118141 , 1.59754453, 1.40185717]))

In [22]:
# first sol:
sol_line_1_raw = [
    '0', '0', '0.0', [-200*AU, human_res[0]*AU, human_res[1]*AU], [human_res[2], 0.0, 0.0], [0.0, 0.0, 0.0]
]

sol_line_2_raw = [
    '0', '0', res['t'][-1], res['y'][0:3,-1].T, res['y'][3:6,-1].T, [0.0, 0.0, 0.0]
]

planetx_id = planetX.gtoc13_id

sol_line_3_raw = [
    planetx_id, '1', res['t'][-1], res['y'][0:3,-1].T, res['y'][3:6,-1].T, rel_v
]

In [23]:
def flatten_list(l):
    out = []
    for v in l:
        if isinstance(v, (int, float, str)):
            out.append(float(v))
        elif isinstance(v, (list, np.ndarray)):
            for y in v:
                out.append(float(y))
    return out

In [24]:
sol_lines = list()
sol_lines.append(flatten_list(sol_line_1_raw))
sol_lines.append(flatten_list(sol_line_2_raw))
sol_lines.append(flatten_list(sol_line_3_raw))

In [25]:
sol_lines

[[0.0,
  0.0,
  0.0,
  -29919574138.200005,
  1625026472.2571514,
  -4265753983.516562,
  3.952118444731843,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0],
 [0.0,
  0.0,
  2209032000.0,
  -20731752896.68304,
  1596950414.5640779,
  -4192053304.1797414,
  4.420677784943144,
  -0.030740580900722576,
  0.08069515030775418,
  0.0,
  0.0,
  0.0],
 [10.0,
  1.0,
  2209032000.0,
  -20731752896.68304,
  1596950414.5640779,
  -4192053304.1797414,
  4.420677784943144,
  -0.030740580900722576,
  0.08069515030775418,
  4.5118141001616,
  1.5975445285723726,
  1.4018571667164514]]

In [26]:
import csv

with open('planetx_intercept.csv', 'w', newline='') as f:
    header = ['#body_id','flag','epoch','rx','ry','rz','vx','vy','vz','ux','uy','uz']
    writer = csv.writer(f)
    writer.writerow(header)
    for l in sol_lines:
        writer.writerow(l)

In [27]:
# calculate time to periapsis
def event2(t,y):
    r = y[0:3]; v = y[3:6]
    return np.dot(r,v)

event2.terminal = True
    
res = solve_ivp(dydt, [0, 200*YEAR], [-200*AU, human_res[0]*AU, human_res[1]*AU, human_res[2], 0.0, 0.0], events=event2, method='DOP853', rtol=1e-12, atol=1e-12)

In [28]:
res['t'] / YEAR

array([0.00000000e+00, 1.07710068e-08, 1.18481075e-07, 1.19558176e-06,
       1.19665886e-05, 1.19676657e-04, 1.19677734e-03, 1.19677842e-02,
       1.19677852e-01, 1.19677853e+00, 1.19677854e+01, 2.92870044e+01,
       4.33577632e+01, 5.74285221e+01, 6.87542704e+01, 8.00800188e+01,
       8.92481170e+01, 9.84162153e+01, 1.05876845e+02, 1.13337475e+02,
       1.19438799e+02, 1.25540124e+02, 1.31097745e+02, 1.36110929e+02,
       1.40636423e+02, 1.44726966e+02, 1.48429183e+02, 1.51784256e+02,
       1.54828628e+02, 1.57594611e+02, 1.60110909e+02, 1.62403081e+02,
       1.64493933e+02, 1.66403865e+02, 1.68151181e+02, 1.69752355e+02,
       1.71222285e+02, 1.72574519e+02, 1.73821481e+02, 1.74974701e+02,
       1.76045074e+02, 1.77043196e+02, 1.77979816e+02, 1.78866493e+02,
       1.79716120e+02, 1.80539037e+02, 1.81183591e+02, 1.81734172e+02,
       1.82225908e+02, 1.82717644e+02, 1.83179950e+02, 1.83593721e+02,
       1.84007493e+02, 1.84391365e+02, 1.84752035e+02, 1.85090096e+02,
      

In [29]:
res['y'][0:3,-1] / AU

array([ 7.1460173 ,  0.92855695, -2.43749598])